In [1]:
# P2PNet for Cell Detection - Imports and Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils import data
import torchvision
from torchvision.models import vgg16_bn, resnet50
import numpy as np
import cv2
import os
import yaml
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from scipy.optimize import linear_sum_assignment
import glob
import json

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 파라미터 로드
with open('utils/args.yaml', errors='ignore') as f:
    params = yaml.safe_load(f)

# 6 클래스 cell type 정의 (HnE 데이터)
class_names = {
    0: "Neutrophil",
    1: "Epithelial",
    2: "Lymphocyte",
    3: "Plasma",
    4: "Eosinophil",
    5: "Connective tissue"
}

num_classes = len(class_names)
print(f"Number of classes: {num_classes}")
print(f"Classes: {list(class_names.values())}")

Device: cuda:0
Number of classes: 6
Classes: ['Neutrophil', 'Epithelial', 'Lymphocyte', 'Plasma', 'Eosinophil', 'Connective tissue']


In [2]:
# P2PNet Model Architecture (Original P2PNet from GitHub)
class P2PNet(nn.Module):
    """
    Point-to-Point Network for Cell Detection (Based on Original P2PNet)
    - Backbone: VGG16-BN or ResNet50
    - Head: Point regression + classification (background + foreground classes)
    - Output: Direct point predictions (no anchors, no heatmaps)
    - num_classes = foreground classes + 1 (background)
    """
    def __init__(self, num_classes=7, backbone='vgg16_bn', row=2, line=2):
        super(P2PNet, self).__init__()
        self.num_classes = num_classes  # 7 = 6 cell types + 1 background
        self.row = row
        self.line = line
        
        # Backbone
        if backbone == 'vgg16_bn':
            vgg = vgg16_bn(pretrained=True)
            self.features = nn.Sequential(*list(vgg.features.children())[:-1])
            in_channels = 512
        elif backbone == 'resnet50':
            resnet = resnet50(pretrained=True)
            self.features = nn.Sequential(
                resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
                resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4
            )
            in_channels = 2048
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")
        
        # Regression head (point coordinates)
        self.reg_head = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, row * line * 2, 1)
        )
        
        # Classification head (원본 P2PNet: background + foreground classes)
        self.cls_head = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, row * line * num_classes, 1)
        )
        
    def forward(self, x):
        batch_size = x.size(0)
        
        # Backbone
        features = self.features(x)
        
        # Regression
        pred_points = self.reg_head(features)
        pred_points = pred_points.permute(0, 2, 3, 1)
        
        # Classification
        pred_logits = self.cls_head(features)
        pred_logits = pred_logits.permute(0, 2, 3, 1)
        
        h_feat, w_feat = pred_points.shape[1], pred_points.shape[2]
        
        # Reshape
        pred_points = pred_points.reshape(batch_size, h_feat * w_feat * self.row * self.line, 2)
        pred_logits = pred_logits.reshape(batch_size, h_feat * w_feat * self.row * self.line, self.num_classes)
        
        pred_points = torch.sigmoid(pred_points)
        
        return pred_points, pred_logits


# 모델 초기화
model = P2PNet(num_classes=num_classes, backbone='vgg16_bn', row=2, line=2).to(device)
print(f"\n✅ P2PNet model created")
print(f"  Backbone: VGG16-BN")
print(f"  Number of classes: {num_classes}")
print(f"  Grid: 2x2 per feature map cell")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)total_params = sum(p.numel() for p in model.parameters())print(f"  Grid: 2x2 per feature map cell")print(f"  Total classes: {num_classes+1} (including background)")print(f"  Foreground classes: {num_classes}")print(f"\n✅ P2PNet model created (Original P2PNet Style)")model = P2PNet(num_classes=num_classes+1, backbone='vgg16_bn', row=2, line=2).to(device)# 모델 초기화 (원본 P2PNet: background class 포함!)

SyntaxError: invalid syntax (2846503976.py, line 85)

In [ ]:
# P2PNet Loss Function (Exact Original P2PNet from GitHub)
class P2PNetLoss(nn.Module):
    def __init__(self, num_classes=7, point_loss_coef=0.0002, eos_coef=0.5):
        super(P2PNetLoss, self).__init__()
        self.num_classes = num_classes  # 7 = 6 foreground + 1 background
        self.point_loss_coef = point_loss_coef
        
        # 원본 P2PNet: background class에 낮은 가중치 (eos_coef)
        empty_weight = torch.ones(self.num_classes)
        empty_weight[0] = eos_coef  # Class 0 (background) 가중치
        self.register_buffer('empty_weight', empty_weight)
    
    def hungarian_matching(self, pred_points, pred_logits, gt_points, gt_classes):
        if len(gt_points) == 0:
            return torch.tensor([]).long(), torch.tensor([]).long()
        
        # Cost 계산 (원본 P2PNet 방식)
        point_cost = torch.cdist(pred_points, gt_points, p=2)  # L2 distance
        
        # 원본: softmax 확률의 negative를 class cost로 사용
        pred_probs = F.softmax(pred_logits, dim=-1)
        # gt_classes는 foreground classes (0-5)이므로 +1 해서 1-6으로 변환
        class_cost = -pred_probs[:, gt_classes + 1]  # +1: background 제외
        
        # 원본 가중치: point_cost * 0.05 + class_cost * 1.0
        cost_matrix = point_cost * 0.05 + class_cost * 1.0
        cost_matrix_np = cost_matrix.detach().cpu().numpy()
        pred_idx, gt_idx = linear_sum_assignment(cost_matrix_np)
        
        return torch.from_numpy(pred_idx).long(), torch.from_numpy(gt_idx).long()
    
    def forward(self, pred_points, pred_logits, targets):
        batch_size = pred_points.size(0)
        device = pred_points.device
        
        total_point_loss = 0
        total_class_loss = 0
        num_matched = 0
        
        for b in range(batch_size):
            gt_points = targets['points'][b]
            gt_classes = targets['classes'][b].long()
            
            valid_mask = (gt_points[:, 0] >= 0) & (gt_points[:, 1] >= 0)
            gt_points = gt_points[valid_mask]
            gt_classes = gt_classes[valid_mask]  # 0-5 (foreground classes)
            
            # Hungarian matching
            pred_idx, gt_idx = self.hungarian_matching(
                pred_points[b], pred_logits[b], gt_points, gt_classes
            )
            
            # Classification loss (원본 P2PNet 방식)
            # Background class = 0, Foreground classes = 1-6
            target_classes = torch.zeros(pred_logits[b].shape[0], dtype=torch.long, device=device)
            
            if len(pred_idx) > 0:
                matched_gt_classes = gt_classes[gt_idx]
                target_classes[pred_idx] = matched_gt_classes + 1  # +1: 1-6으로 변환
                
                # Point Loss: MSE (matched만)
                matched_pred_points = pred_points[b][pred_idx]
                matched_gt_points = gt_points[gt_idx]
                point_loss = F.mse_loss(matched_pred_points, matched_gt_points, reduction='sum')
                total_point_loss += point_loss
                num_matched += len(pred_idx)
            
            # Classification loss (전체 predictions에 대해)
            # 원본: Cross Entropy with empty_weight
            class_loss = F.cross_entropy(
                pred_logits[b], 
                target_classes, 
                weight=self.empty_weight,
                reduction='mean'
            )
            total_class_loss += class_loss
        
        if num_matched > 0:
            total_point_loss = total_point_loss / num_matched
        else:
            total_point_loss = torch.tensor(0.0, device=device)
        
        total_class_loss = total_class_loss / batch_size
        
        # 원본 P2PNet 가중치 적용
        total_loss = (self.point_loss_coef * total_point_loss + 
                     1.0 * total_class_loss)
        
        loss_dict = {
            'total': total_loss.item(),
            'point': total_point_loss.item(),
            'class': total_class_loss.item(),
            'num_matched': num_matched
        }
        
        return total_loss, loss_dict


# Loss function 초기화 (원본 P2PNet 가중치 적용!)
criterion = P2PNetLoss(
    num_classes=num_classes, 
    point_loss_coef=0.0002,  # 원본처럼 매우 작게!
    class_loss_coef=1.0
).to(device)

print(f"\n✅ P2PNet loss function created (Based on Original P2PNet)")
print(f"  Point loss coefficient: 0.0002 (원본 방식 - MSE)")
print(f"  Class loss coefficient: 1.0")
print(f"  Objectness loss coefficient: 1.0")
print(f"  Matching: Hungarian algorithm (L2 distance)")
print(f"  ⚠️  Loss 가중치를 원본 P2PNet에 맞게 수정했습니다!")


In [ ]:
# Data Loading (HnE format - JSON)
input_size = 512
label_dir = '../../data/HnE_cell_detect/total_data/labels/'
image_dir = '../../data/HnE_cell_detect/total_data/images/'

label_files = sorted(glob.glob(os.path.join(label_dir, '*.json')))

image_filenames = []
labels = []

print("📂 Loading labels...")
for i in tqdm(range(len(label_files))):
    label_file = label_files[i]
    
    with open(label_file) as f:
        data_json = json.load(f)
    
    img_path = os.path.join(image_dir, data_json['file_name'])
    
    if os.path.exists(img_path):
        image_filenames.append(img_path)
        
        centers = []
        classes = []
        
        # JSON 형식: data_json["cordinates"] = [[class_id, y, x, h, w], ...]
        for coord in data_json["cordinates"]:
            if len(coord) < 5:
                continue
                
            class_id = int(coord[0]) - 1  # HnE 데이터는 1부터 시작하므로 -1
            y = coord[1]
            x= coord[2]
            h = coord[3]
            w = coord[4]
            
            # 너무 큰 박스 제외
            if h > 50 or w > 50:
                continue
            
            # 중심점 좌표 (픽셀 단위)
            centers.append([x+w//2, y+h//2])
            classes.append(class_id)
        
        if len(centers) > 0:
            labels.append({
                'points': np.array(centers, dtype=np.float32),
                'classes': np.array(classes, dtype=np.int64)
            })
        else:
            image_filenames.pop()

print(f"✅ Loaded {len(image_filenames)} images with labels")

print("\n📷 Loading images...")
images = []
for i in tqdm(range(len(image_filenames))):
    image = cv2.imread(image_filenames[i])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    images.append(image)

print(f"✅ Loaded {len(images)} images")
print(f"  Image shape: {images[0].shape}")
print(f"  Label example - Points: {labels[0]['points'].shape}, Classes: {labels[0]['classes'].shape}")

# 데이터 검증
print(f"\n📊 Data validation:")
all_points = np.concatenate([l['points'] for l in labels])
all_classes = np.concatenate([l['classes'] for l in labels])
print(f"  Total points: {len(all_points):,}")
print(f"  Point range: [{all_points.min():.2f}, {all_points.max():.2f}]")
print(f"  Class distribution:")
for i in range(num_classes):
    count = (all_classes == i).sum()
    print(f"    {class_names[i]}: {count:,} ({count/len(all_classes)*100:.1f}%)")
print(f"  Avg points per image: {len(all_points)/len(images):.1f}")
print(f"  Max points in single image: {max([len(l['points']) for l in labels])}")

In [ ]:
# Custom Dataset for P2PNet
class P2PNetDataset(data.Dataset):
    def __init__(self, images, labels, img_size=512, augment=False, max_points=500):
        self.images = images
        self.labels = labels
        self.img_size = img_size
        self.augment = augment
        self.max_points = max_points
        self.n = len(self.images)
    
    def __len__(self):
        return self.n
    
    def __getitem__(self, index):
        image = self.images[index].copy()
        points = self.labels[index]['points'].copy()
        classes = self.labels[index]['classes'].copy()
        
        h, w = image.shape[:2]
        
        # 크롭 또는 패딩하여 정확히 img_size x img_size로 만들기
        # 1. Height 처리
        if h > self.img_size:
            # 크롭
            if self.augment:
                h_start = random.randint(0, h - self.img_size)
            else:
                h_start = 0
            image = image[h_start:h_start + self.img_size, :]
            # 포인트 좌표 조정
            points[:, 1] -= h_start
        elif h < self.img_size:
            # 패딩
            pad_h = self.img_size - h
            pad_top = 0
            pad_bottom = pad_h
            image = cv2.copyMakeBorder(image, pad_top, pad_bottom, 0, 0, 
                                      cv2.BORDER_CONSTANT, value=[255, 255, 255])
        
        # 2. Width 처리
        h, w = image.shape[:2]  # 업데이트된 크기
        if w > self.img_size:
            # 크롭
            if self.augment:
                w_start = random.randint(0, w - self.img_size)
            else:
                w_start = 0
            image = image[:, w_start:w_start + self.img_size]
            # 포인트 좌표 조정
            points[:, 0] -= w_start
        elif w < self.img_size:
            # 패딩
            pad_w = self.img_size - w
            pad_left = 0
            pad_right = pad_w
            image = cv2.copyMakeBorder(image, 0, 0, pad_left, pad_right,
                                      cv2.BORDER_CONSTANT, value=[255, 255, 255])
        
        # 이제 이미지는 정확히 img_size x img_size
        h, w = image.shape[:2]
        assert h == self.img_size and w == self.img_size, f"Image size mismatch: {h}x{w} != {self.img_size}x{self.img_size}"
        
        # 포인트 필터링 (이미지 범위 내에 있는 것만)
        valid_mask = (points[:, 0] >= 0) & (points[:, 0] < self.img_size) & \
                    (points[:, 1] >= 0) & (points[:, 1] < self.img_size)
        points = points[valid_mask]
        classes = classes[valid_mask]
        
        if self.augment:
            # Geometric augmentation (영향: 포인트 좌표)
            if random.random() < 0.5:
                image = np.fliplr(image).copy()
                points[:, 0] = w - points[:, 0]
            
            if random.random() < 0.5:
                image = np.flipud(image).copy()
                points[:, 1] = h - points[:, 1]
            
            # Style augmentation (영향: 이미지만, 포인트는 영향 없음)
            # 병원 데이터셋 적용을 위한 다양한 스타일 변환
            
            # 1. 밝기 조정 (Brightness)
            if random.random() < 0.5:
                brightness_factor = random.uniform(0.7, 1.3)
                image = np.clip(image * brightness_factor, 0, 255).astype(np.uint8)
            
            # 2. 대비 조정 (Contrast)
            if random.random() < 0.5:
                contrast_factor = random.uniform(0.7, 1.3)
                mean = image.mean(axis=(0, 1), keepdims=True)
                image = np.clip((image - mean) * contrast_factor + mean, 0, 255).astype(np.uint8)
            
            # 3. 채도 조정 (Saturation) - HSV 색공간에서
            if random.random() < 0.3:
                hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
                saturation_factor = random.uniform(0.7, 1.3)
                hsv[:, :, 1] = np.clip(hsv[:, :, 1] * saturation_factor, 0, 255)
                image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
            
            # 4. 색조 조정 (Hue shift) - 미세한 조정
            if random.random() < 0.3:
                hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
                hue_shift = random.uniform(-10, 10)
                hsv[:, :, 0] = (hsv[:, :, 0] + hue_shift) % 180
                image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
            
            # 5. 가우시안 블러 (Gaussian Blur)
            if random.random() < 0.2:
                kernel_size = random.choice([3, 5])
                image = cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)
            
            # 6. 가우시안 노이즈 (Gaussian Noise)
            if random.random() < 0.2:
                noise_sigma = random.uniform(3, 10)
                noise = np.random.normal(0, noise_sigma, image.shape)
                image = np.clip(image + noise, 0, 255).astype(np.uint8)
            
            # 7. 감마 보정 (Gamma Correction) - HnE 염색 농도 차이 시뮬레이션
            if random.random() < 0.3:
                gamma = random.uniform(0.8, 1.2)
                inv_gamma = 1.0 / gamma
                table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype(np.uint8)
                image = cv2.LUT(image, table)
        
        # 포인트 정규화 (0~1 범위)
        points[:, 0] = points[:, 0] / w
        points[:, 1] = points[:, 1] / h
        
        # 이미지 정규화
        image = image.astype(np.float32) / 255.0
        image = image.transpose((2, 0, 1))
        
        # 패딩
        num_points = len(points)
        if num_points < self.max_points:
            padded_points = np.full((self.max_points, 2), -1.0, dtype=np.float32)
            padded_classes = np.full((self.max_points,), -1, dtype=np.int64)
            
            padded_points[:num_points] = points
            padded_classes[:num_points] = classes
            
            points = padded_points
            classes = padded_classes
        else:
            points = points[:self.max_points]
            classes = classes[:self.max_points]
        
        return (torch.from_numpy(image).float(),
                torch.from_numpy(points).float(),
                torch.from_numpy(classes).long())


def collate_fn_p2pnet(batch):
    images, points, classes = zip(*batch)
    
    images = torch.stack(images, dim=0)
    points = torch.stack(points, dim=0)
    classes = torch.stack(classes, dim=0)
    
    targets = {
        'points': points,
        'classes': classes
    }
    
    return images, targets


# Train/Val split
train_images, val_images, train_labels, val_labels = train_test_split(
    images, labels, test_size=0.1, random_state=42, shuffle=True
)

train_dataset = P2PNetDataset(train_images, train_labels, augment=True, max_points=500)
val_dataset = P2PNetDataset(val_images, val_labels, augment=False, max_points=500)

print(f"\n📊 Dataset split:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Max points per image: 500")

batch_size = 8
train_loader = data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=4, collate_fn=collate_fn_p2pnet, pin_memory=True
)
val_loader = data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=4, collate_fn=collate_fn_p2pnet, pin_memory=True
)

print(f"\n📦 Dataloaders created:")
print(f"  Batch size: {batch_size}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

In [ ]:
# Visualization function
def visualize_p2pnet_sample(dataset, index=0):
    image_tensor, points_tensor, classes_tensor = dataset[index]
    
    image = image_tensor.numpy().transpose(1, 2, 0)
    points = points_tensor.numpy()
    classes = classes_tensor.numpy()
    
    valid_mask = (points[:, 0] >= 0) & (points[:, 1] >= 0)
    points = points[valid_mask]
    classes = classes[valid_mask]
    
    h, w = image.shape[:2]
    points_pixel = points.copy()
    points_pixel[:, 0] *= w
    points_pixel[:, 1] *= h
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(image)
    
    colors = ['orange', 'green', 'red', 'skyblue', 'blue', 'yellow']
    
    for i in range(len(points_pixel)):
        x, y = points_pixel[i]
        class_id = int(classes[i])
        color = colors[class_id] if class_id < len(colors) else 'white'
        
        circle = plt.Circle((x, y), 3, color=color, fill=True, alpha=0.7)
        ax.add_patch(circle)
    
    ax.set_title(f'Sample {index} - Total points: {len(points)}', fontsize=14, fontweight='bold')
    ax.axis('off')
    
    legend_elements = [
        patches.Patch(color=colors[i], label=f'{class_names[i]}: {sum(classes==i)}')
        for i in range(num_classes) if sum(classes==i) > 0
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Sample {index} statistics:")
    print(f"  Total points: {len(points)}")
    for i in range(num_classes):
        count = sum(classes == i)
        if count > 0:
            print(f"  {class_names[i]}: {count}")

print("🖼️ Visualizing training sample...")
visualize_p2pnet_sample(train_dataset, index=19)

In [ ]:
# Training Setup
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-4)

epochs = 10000
save_dir = '../../model/HnE_cell_detection/p2pnet/'
os.makedirs(save_dir, exist_ok=True)

train_losses = []
val_point_errors = []
val_class_accs = []
val_recalls = []
val_precisions = []
best_val_f1 = 0

print(f"\n🚀 Training setup:")
print(f"  Epochs: {epochs}")
print(f"  Optimizer: AdamW (lr=1e-4, wd=1e-4)")
print(f"  Scheduler: CosineAnnealingWarmRestarts")
print(f"  Save directory: {save_dir}")

# 체크포인트 불러오기 (선택사항)
# checkpoint_path = os.path.join(save_dir, 'last_model.pt')
# if os.path.exists(checkpoint_path):
#     checkpoint = torch.load(checkpoint_path, map_location=device)
#     model.load_state_dict(checkpoint['model_state_dict'])
#     optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#     scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
#     print(f"✅ Checkpoint loaded from {checkpoint_path}")

In [ ]:
# Training Loop
print("\n" + "="*80)
print("🚀 Starting P2PNet Training")
print("="*80)

for epoch in range(epochs):
    # TRAINING
    model.train()
    train_loss = 0
    train_point_loss = 0
    train_class_loss = 0
    train_obj_loss = 0
    train_matched = 0
    
    train_pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                     desc=f'Epoch {epoch+1}/{epochs} [Train]')
    
    for batch_idx, (images, targets) in train_pbar:
        images = images.to(device)
        targets = {k: v.to(device) for k, v in targets.items()}
        
        pred_points, pred_logits, pred_obj = model(images)
        
        if epoch == 0 and batch_idx == 0:
            print(f"\n📊 First batch info:")
            print(f"  Input shape: {images.shape}")
            print(f"  Pred points: {pred_points.shape}")
            print(f"  Pred logits: {pred_logits.shape}")
            print(f"  Pred objectness: {pred_obj.shape}")
            print(f"  GT points: {targets['points'].shape}")
            
            valid_mask = (targets['points'][0, :, 0] >= 0) & (targets['points'][0, :, 1] >= 0)
            num_valid = valid_mask.sum().item()
            print(f"  Valid GT points: {num_valid}")
            
            obj_scores = torch.sigmoid(pred_obj[0]).detach().cpu().numpy()
            print(f"  Objectness range: [{obj_scores.min():.4f}, {obj_scores.max():.4f}]")
            print(f"  Objectness > 0.5: {(obj_scores > 0.5).sum()} / {len(obj_scores)}")
        
        loss, loss_dict = criterion(pred_points, pred_logits, pred_obj, targets)
        
        if epoch == 0 and batch_idx == 0:
            print(f"\n💰 First batch loss (Fixed - Original P2PNet):")
            print(f"  Total: {loss_dict['total']:.4f}")
            print(f"  Point: {loss_dict['point']:.4f} (coef=0.0002, MSE)")
            print(f"  Class: {loss_dict['class']:.4f} (coef=1.0)")
            print(f"  Obj: {loss_dict['obj']:.4f} (coef=1.0)")
            print(f"  Matched: {loss_dict['num_matched']}\n")
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss_dict['total']
        train_point_loss += loss_dict['point']
        train_class_loss += loss_dict['class']
        train_obj_loss += loss_dict['obj']
        train_matched += loss_dict['num_matched']
        
        memory = f'{torch.cuda.memory_reserved() / 1E9:.2f}G'
        train_pbar.set_postfix({
            'loss': f"{loss_dict['total']:.4f}",
            'pt': f"{loss_dict['point']:.4f}",
            'cls': f"{loss_dict['class']:.4f}",
            'obj': f"{loss_dict['obj']:.4f}",
            'mem': memory
        })
    
    num_batches = len(train_loader)
    avg_train_loss = train_loss / num_batches
    avg_train_point_loss = train_point_loss / num_batches
    avg_train_class_loss = train_class_loss / num_batches
    avg_train_obj_loss = train_obj_loss / num_batches
    avg_train_matched = train_matched / num_batches
    train_losses.append(avg_train_loss)
    
    # VALIDATION (Fixed: Spatial Distance-based Precision/Recall)
    model.eval()
    val_point_error = 0
    val_class_correct = 0
    val_total = 0
    val_tp = 0  # True Positives
    val_fp = 0  # False Positives  
    val_fn = 0  # False Negatives
    val_samples = 0
    
    distance_threshold = 0.02  # normalized [0-1] distance (약 10픽셀/512 = 0.02)
    
    val_pbar = tqdm(enumerate(val_loader), total=len(val_loader),
                   desc=f'Epoch {epoch+1}/{epochs} [Val]')
    
    with torch.no_grad():
        for batch_idx, (images, targets) in val_pbar:
            images = images.to(device)
            targets = {k: v.to(device) for k, v in targets.items()}
            
            pred_points, pred_logits, pred_obj = model(images)
            
            batch_size = images.size(0)
            for b in range(batch_size):
                gt_points = targets['points'][b]
                gt_classes = targets['classes'][b].long()
                
                valid_mask = (gt_points[:, 0] >= 0) & (gt_points[:, 1] >= 0)
                gt_points = gt_points[valid_mask]
                gt_classes = gt_classes[valid_mask]
                
                num_gt = len(gt_points)
                
                # Objectness threshold로 예측 필터링
                obj_scores = torch.sigmoid(pred_obj[b])
                obj_mask = obj_scores > 0.5
                
                if obj_mask.sum() == 0:
                    # 예측이 없으면 FN만 증가
                    val_fn += num_gt
                    if num_gt > 0:
                        val_samples += 1
                    continue
                
                filtered_pred_points = pred_points[b][obj_mask]
                filtered_pred_logits = pred_logits[b][obj_mask]
                num_pred = len(filtered_pred_points)
                
                if num_gt == 0:
                    # GT가 없으면 모든 예측이 FP
                    val_fp += num_pred
                    continue
                
                # Hungarian matching
                point_cost = torch.cdist(filtered_pred_points, gt_points, p=2)
                pred_probs = F.softmax(filtered_pred_logits, dim=-1)
                class_cost = -pred_probs[:, gt_classes]
                cost_matrix = (point_cost * 0.05 + class_cost * 1.0).detach().cpu().numpy()
                
                pred_idx, gt_idx = linear_sum_assignment(cost_matrix)
                
                # Spatial distance로 TP/FP/FN 계산
                matched_pred_points = filtered_pred_points[pred_idx]
                matched_gt_points = gt_points[gt_idx]
                distances = torch.sqrt(((matched_pred_points - matched_gt_points) ** 2).sum(dim=1))
                
                # Distance threshold 내에 있으면 TP
                tp_mask = distances < distance_threshold
                tp_count = tp_mask.sum().item()
                
                val_tp += tp_count
                val_fp += (num_pred - tp_count)  # 매칭되지 않거나 distance가 먼 예측
                val_fn += (num_gt - tp_count)    # 매칭되지 않은 GT
                
                # Point error (matched only)
                if len(pred_idx) > 0:
                    point_error = torch.abs(matched_pred_points - matched_gt_points).mean()
                    val_point_error += point_error.item()
                
                # Class accuracy (TP만)
                if tp_count > 0:
                    matched_pred_classes = filtered_pred_logits[pred_idx][tp_mask].argmax(dim=-1)
                    matched_gt_classes = gt_classes[gt_idx][tp_mask]
                    correct = (matched_pred_classes == matched_gt_classes).sum().item()
                    val_class_correct += correct
                    val_total += tp_count
                
                val_samples += 1
    
    avg_val_point_error = val_point_error / val_samples if val_samples > 0 else 0
    avg_val_class_acc = val_class_correct / val_total if val_total > 0 else 0
    
    # Spatial Distance-based Precision/Recall
    avg_val_precision = val_tp / (val_tp + val_fp) if (val_tp + val_fp) > 0 else 0
    avg_val_recall = val_tp / (val_tp + val_fn) if (val_tp + val_fn) > 0 else 0
    avg_val_f1 = 2 * avg_val_precision * avg_val_recall / (avg_val_precision + avg_val_recall + 1e-8)
    
    val_point_errors.append(avg_val_point_error)
    val_class_accs.append(avg_val_class_acc)
    val_recalls.append(avg_val_recall)
    val_precisions.append(avg_val_precision)
    
    scheduler.step()
    
    # LOGGING
    print(f"\n{'='*80}")
    print(f"Epoch {epoch+1}/{epochs} Summary:")
    print(f"  Train Loss: {avg_train_loss:.4f} (Pt: {avg_train_point_loss:.4f}, Cls: {avg_train_class_loss:.4f}, Obj: {avg_train_obj_loss:.4f})")
    print(f"  Train Matched: {avg_train_matched:.1f} points/batch")
    print(f"  Val Point Error: {avg_val_point_error:.4f}")
    print(f"  Val Class Acc: {avg_val_class_acc:.4f}")
    print(f"  Val Precision: {avg_val_precision:.4f} (TP/{val_tp}, FP/{val_fp})")
    print(f"  Val Recall: {avg_val_recall:.4f} (TP/{val_tp}, FN/{val_fn})")
    print(f"  Val F1: {avg_val_f1:.4f} 🎯")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"{'='*80}\n")
    
    # SAVE CHECKPOINT
    if avg_val_f1 > best_val_f1 and avg_val_f1 > 0:
        best_val_f1 = avg_val_f1
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': avg_train_loss,
            'val_point_error': avg_val_point_error,
            'val_class_acc': avg_val_class_acc,
            'best_val_f1': best_val_f1
        }
        torch.save(checkpoint, os.path.join(save_dir, 'best_model.pt'))
        print(f"🎉 New best model saved! f1: {best_val_f1:.4f}\n")
    
    last_checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': avg_train_loss,
        'val_point_error': avg_val_point_error,
        'val_class_acc': avg_val_class_acc,
        'val_f1': avg_val_f1
        
    }
    torch.save(last_checkpoint, os.path.join(save_dir, 'last_model.pt'))
    
    # PLOT PROGRESS (100 에폭마다)
    if (epoch + 1) % 100 == 0:
        fig, axes = plt.subplots(2, 2, figsize=(18, 12))
        
        axes[0,0].plot(train_losses, 'b-')
        axes[0,0].set_title('Training Loss')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].grid(True)
        
        axes[0,1].plot(val_point_errors, 'r-')
        axes[0,1].set_title('Val Point Error')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].grid(True)
        
        axes[1,0].plot(val_class_accs, 'g-')
        axes[1,0].set_title('Val Class Accuracy')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].grid(True)
        
        axes[1,1].plot(val_recalls, 'c-', label='Recall')
        axes[1,1].plot(val_precisions, 'm-', label='Precision')
        axes[1,1].set_title('Val Recall & Precision')
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].legend()
        axes[1,1].grid(True)
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f'training_progress_epoch_{epoch+1}.png'), dpi=150)
        plt.close()
        print(f"📊 Training progress saved: training_progress_epoch_{epoch+1}.png\n")
    
    # VISUALIZE PREDICTIONS (10 에폭마다)
    if (epoch + 1) % 10 == 0:
        model.eval()
        sample_idx = random.randint(0, len(val_dataset)-1)
        image_tensor, gt_points_tensor, gt_classes_tensor = val_dataset[sample_idx]
        
        valid_mask = (gt_points_tensor[:, 0] >= 0) & (gt_points_tensor[:, 1] >= 0)
        gt_points = gt_points_tensor[valid_mask].numpy()
        gt_classes = gt_classes_tensor[valid_mask].numpy()
        
        with torch.no_grad():
            image_batch = image_tensor.unsqueeze(0).to(device)
            pred_points, pred_logits, pred_obj = model(image_batch)
            pred_points_np = pred_points[0].cpu().numpy()
            pred_classes_np = pred_logits[0].argmax(dim=-1).cpu().numpy()
            pred_scores = F.softmax(pred_logits[0], dim=-1).max(dim=-1)[0].cpu().numpy()
            pred_obj_scores = torch.sigmoid(pred_obj[0]).cpu().numpy()
        
        obj_threshold = 0.5
        conf_threshold = 0.5
        obj_mask = pred_obj_scores > obj_threshold
        pred_points_np = pred_points_np[obj_mask]
        pred_classes_np = pred_classes_np[obj_mask]
        pred_scores = pred_scores[obj_mask]
        
        if len(pred_points_np) > 0:
            conf_mask = pred_scores > conf_threshold
            pred_points_np = pred_points_np[conf_mask]
            pred_classes_np = pred_classes_np[conf_mask]
        
        image = image_tensor.numpy().transpose(1, 2, 0)
        h, w = image.shape[:2]
        
        gt_points_pixel = gt_points.copy()
        gt_points_pixel[:, 0] *= w
        gt_points_pixel[:, 1] *= h
        
        pred_points_pixel = pred_points_np.copy()
        pred_points_pixel[:, 0] *= w
        pred_points_pixel[:, 1] *= h
        
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        colors = ['orange', 'green', 'red', 'skyblue', 'blue', 'yellow']
        
        # Ground Truth
        axes[0].imshow(image)
        for i in range(len(gt_points_pixel)):
            x, y = gt_points_pixel[i]
            class_id = int(gt_classes[i])
            color = colors[class_id] if class_id < len(colors) else 'white'
            circle = plt.Circle((x, y), 4, color=color, fill=True, alpha=0.8)
            axes[0].add_patch(circle)
        axes[0].set_title(f'Ground Truth ({len(gt_points_pixel)} points)', fontsize=14, fontweight='bold')
        axes[0].axis('off')
        
        # Predictions
        axes[1].imshow(image)
        for i in range(len(pred_points_pixel)):
            x, y = pred_points_pixel[i]
            class_id = int(pred_classes_np[i])
            color = colors[class_id] if class_id < len(colors) else 'white'
            circle = plt.Circle((x, y), 4, color=color, fill=True, alpha=0.8)
            axes[1].add_patch(circle)
        
        axes[1].set_title(f'Predictions ({len(pred_points_pixel)} points)', fontsize=14, fontweight='bold')
        axes[1].axis('off')
        
        legend_elements = [
            patches.Patch(color=colors[i], label=class_names[i])
            for i in range(num_classes)
        ]
        fig.legend(handles=legend_elements, loc='lower center', ncol=num_classes,
                  bbox_to_anchor=(0.5, -0.05), fontsize=12)
        
        plt.tight_layout()
        plt.subplots_adjust(bottom=0.1)
        plt.savefig(os.path.join(save_dir, f'prediction_epoch_{epoch+1}.png'), dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"📸 Prediction visualization saved: prediction_epoch_{epoch+1}.png")
        print(f"  GT: {len(gt_points_pixel)} points, Pred: {len(pred_points_pixel)} points\n")

print("\n" + "="*80)
print("🎯 Training Complete!")
print(f"  Best Val f1: {best_val_f1:.4f}")
print(f"  Models saved to: {save_dir}")
print("="*80)


In [ ]:
# Evaluation and Visualization
def nms_points(points, scores, classes, nms_threshold=10.0):
    """NMS for point predictions"""
    if len(points) == 0:
        return np.array([])
    
    points_pixel = points * 512
    sorted_indices = np.argsort(-scores)
    
    keep = []
    while len(sorted_indices) > 0:
        current_idx = sorted_indices[0]
        keep.append(current_idx)
        
        if len(sorted_indices) == 1:
            break
        
        current_point = points_pixel[current_idx]
        remaining_points = points_pixel[sorted_indices[1:]]
        distances = np.sqrt(np.sum((remaining_points - current_point) ** 2, axis=1))
        
        current_class = classes[current_idx]
        remaining_classes = classes[sorted_indices[1:]]
        same_class = (remaining_classes == current_class)
        
        keep_mask = (distances > nms_threshold) | (~same_class)
        sorted_indices = sorted_indices[1:][keep_mask]
    
    return np.array(keep)


def visualize_predictions(model, dataset, index=0, conf_threshold=0.5, obj_threshold=0.5, nms_threshold=10.0):
    """Visualize ground truth and predictions side by side"""
    model.eval()
    
    image_tensor, gt_points_tensor, gt_classes_tensor = dataset[index]
    
    valid_mask = (gt_points_tensor[:, 0] >= 0) & (gt_points_tensor[:, 1] >= 0)
    gt_points = gt_points_tensor[valid_mask].numpy()
    gt_classes = gt_classes_tensor[valid_mask].numpy()
    
    with torch.no_grad():
        image_batch = image_tensor.unsqueeze(0).to(device)
        pred_points, pred_logits, pred_obj = model(image_batch)
        pred_points = pred_points[0].cpu().numpy()
        pred_classes = pred_logits[0].argmax(dim=-1).cpu().numpy()
        pred_scores = F.softmax(pred_logits[0], dim=-1).max(dim=-1)[0].cpu().numpy()
        pred_obj_scores = torch.sigmoid(pred_obj[0]).cpu().numpy()
    
    print(f"  Total predictions: {len(pred_points)}")
    
    obj_mask = pred_obj_scores > obj_threshold
    pred_points = pred_points[obj_mask]
    pred_classes = pred_classes[obj_mask]
    pred_scores = pred_scores[obj_mask]
    pred_obj_scores = pred_obj_scores[obj_mask]
    
    print(f"  After objectness (>{obj_threshold}): {len(pred_points)}")
    
    conf_mask = pred_scores > conf_threshold
    pred_points = pred_points[conf_mask]
    pred_classes = pred_classes[conf_mask]
    pred_scores = pred_scores[conf_mask]
    
    print(f"  After confidence (>{conf_threshold}): {len(pred_points)}")
    
    if len(pred_points) > 0:
        keep_indices = nms_points(pred_points, pred_scores, pred_classes, nms_threshold=nms_threshold)
        pred_points = pred_points[keep_indices]
        pred_classes = pred_classes[keep_indices]
        pred_scores = pred_scores[keep_indices]
        
        print(f"  After NMS ({nms_threshold}px): {len(pred_points)}")
    
    image = image_tensor.numpy().transpose(1, 2, 0)
    h, w = image.shape[:2]
    
    gt_points_pixel = gt_points.copy()
    gt_points_pixel[:, 0] *= w
    gt_points_pixel[:, 1] *= h
    
    pred_points_pixel = pred_points.copy()
    pred_points_pixel[:, 0] *= w
    pred_points_pixel[:, 1] *= h
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    colors = ['orange', 'green', 'red', 'skyblue', 'blue', 'yellow']
    
    # Ground Truth
    axes[0].imshow(image)
    for i in range(len(gt_points_pixel)):
        x, y = gt_points_pixel[i]
        class_id = int(gt_classes[i])
        color = colors[class_id] if class_id < len(colors) else 'white'
        circle = plt.Circle((x, y), 4, color=color, fill=True, alpha=0.8)
        axes[0].add_patch(circle)
    axes[0].set_title(f'Ground Truth ({len(gt_points_pixel)} points)', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # Predictions
    axes[1].imshow(image)
    for i in range(len(pred_points_pixel)):
        x, y = pred_points_pixel[i]
        class_id = int(pred_classes[i])
        color = colors[class_id] if class_id < len(colors) else 'white'
        circle = plt.Circle((x, y), 4, color=color, fill=True, alpha=0.8)
        axes[1].add_patch(circle)
    
    axes[1].set_title(f'Predictions ({len(pred_points_pixel)} points)', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    legend_elements = [
        patches.Patch(color=colors[i], label=class_names[i])
        for i in range(num_classes)
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=num_classes,
              bbox_to_anchor=(0.5, -0.05), fontsize=12)
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.1)
    plt.show()
    
    print(f"\n📊 Sample {index} statistics:")
    print(f"  Ground Truth: {len(gt_points_pixel)} points")
    print(f"  Predictions: {len(pred_points_pixel)} points")


# Load best model
checkpoint_path = os.path.join(save_dir, 'best_model.pt')
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Best model loaded (Epoch {checkpoint['epoch']+1})")
    print(f"  Val Point Error: {checkpoint.get('val_point_error', 'N/A'):.4f}")

print("\n🖼️ Visualizing predictions...")
sample_idx = random.randint(0, len(val_dataset)-1)
visualize_predictions(model, val_dataset, index=sample_idx, conf_threshold=0.5, obj_threshold=0.5, nms_threshold=10.0)